# ArchiLector · Entrenamiento del verificador de volteo

IA principal del proyecto (Documento de Diseño v2.2, Bloque IV). Después de cada volteo el robot
compara la foto de antes con la de después y decide cuál de cuatro cosas pasó:

| Clase | Qué hace el robot |
|---|---|
| `pagina_nueva` | sigue |
| `misma_pagina` | `RETRY,40` (hasta 2 veces) |
| `dos_hojas` | `BACK` y verifica de nuevo |
| `hoja_doblada` | se detiene y pide revisar la hoja |

**Pasos (sección 4.3):** grabar → provocar errores → etiquetar en el panel web → dividir 70/15/15 →
entrenar MobileNetV2 → evaluar (F1, matriz de confusión, comparación con la regla de píxeles) →
exportar a TensorFlow Lite 8 bits → copiar a la Raspberry Pi.

**Metas (sección 4.4):** F1 promedio ≥ 0,90 · recall de `dos_hojas` ≥ 0,95 · mejora clara frente a la regla de píxeles.

Use un entorno con GPU: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*.

## 1. Cargar los datos etiquetados

En el panel web del robot: **Etiquetar → Exportar verificador (.zip)**. El ZIP trae una carpeta
por clase con las imágenes compuestas (mosaico 224 × 224 de antes, después, canto y diferencia,
generado por `archilector/ia/compuesto.py`) y `meta.csv` con el libro y lo que dijo la regla de píxeles.

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

subido = files.upload()  # elegir verificador.zip (o montar Google Drive y apuntar a la ruta)
CARPETA = Path("datos_verificador")
zipfile.ZipFile(next(iter(subido))).extractall(CARPETA)

In [ ]:
CLASES = ["pagina_nueva", "misma_pagina", "dos_hojas", "hoja_doblada"]  # mismo orden que en la Pi
COLUMNA = "clase"
NOMBRE = "verificador"

import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit

def dividir_por_grupo(df, semilla=42):
    """70 % entrenamiento, 15 % validación y 15 % prueba, separando por grupo.

    El grupo es el libro. Con pocos libros (3 en la meta del proyecto) no se
    puede dejar un libro entero para cada conjunto, así que se agrupan bloques
    de 25 capturas consecutivas del mismo libro: páginas casi iguales nunca
    quedan repartidas entre entrenamiento y prueba.
    """
    df = df.copy()
    if df["libro"].nunique() >= 6:
        df["grupo"] = df["libro"]
    else:
        df["grupo"] = df["libro"] + "_" + (df["id"] // 25).astype(str)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=semilla)
    i_ent, i_resto = next(gss.split(df, groups=df["grupo"]))
    resto = df.iloc[i_resto]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=semilla)
    i_val, i_pru = next(gss2.split(resto, groups=resto["grupo"]))
    return df.iloc[i_ent], resto.iloc[i_val], resto.iloc[i_pru]

meta = pd.read_csv(CARPETA / "meta.csv")
meta["id"] = meta["archivo"].str.extract(r"__(\d+)\.jpg$").astype(int)
print(meta[COLUMNA].value_counts(), "\n")
print("Libros:", meta["libro"].unique())
faltan = {c: 300 - n for c, n in meta[COLUMNA].value_counts().items() if n < 300}
if faltan:
    print("Aviso: la meta es 300 pares por clase. Faltan:", faltan)

ent, val, pru = dividir_por_grupo(meta)
for nombre, d in (("entrenamiento", ent), ("validación", val), ("prueba", pru)):
    print(f"{nombre:>13}: {len(d):4d} pares ·", dict(d[COLUMNA].value_counts()))

## 2. Ver algunos ejemplos

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
fig, ejes = plt.subplots(1, len(CLASES), figsize=(4 * len(CLASES), 4))
for eje, c in zip(ejes, CLASES):
    fila = meta[meta[COLUMNA] == c]
    if len(fila):
        eje.imshow(Image.open(CARPETA / fila.sample(1, random_state=1)["archivo"].iloc[0]))
    eje.set_title(c); eje.axis("off")
plt.show()

## 3. Entrenar MobileNetV2 por transferencia de aprendizaje

Etapa 1: solo la capa nueva de 4 clases. Etapa 2: también las últimas 30 capas, con una tasa de aprendizaje baja.

In [ ]:
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

TAM = 224
LOTE = 32

def cargar(ruta):
    img = tf.io.decode_jpeg(tf.io.read_file(ruta), channels=3)
    return tf.image.resize(img, (TAM, TAM))  # píxeles 0..255: el modelo se reescala solo

aumentar = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.02),          # rotación leve
    tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255)),  # cambios de luz
    tf.keras.layers.RandomContrast(0.15),
], name="aumentos")

def nitidez_aleatoria(img):
    # Desenfoca o enfoca un poco (tolerancia a la nitidez de la cámara)
    k = tf.constant([[1, 2, 1], [2, 4, 2], [1, 2, 1]], tf.float32) / 16.0
    k = tf.tile(k[:, :, None, None], [1, 1, 3, 1])
    suave = tf.nn.depthwise_conv2d(img[None], k, [1, 1, 1, 1], "SAME")[0]
    a = tf.random.uniform([], -0.6, 0.6)
    return tf.clip_by_value(img + a * (img - suave), 0, 255)

def conjunto(df, entrenar):
    rutas = [str(CARPETA / a) for a in df["archivo"]]
    etiquetas = [CLASES.index(c) for c in df[COLUMNA]]
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))
    if entrenar:
        ds = ds.shuffle(len(rutas), seed=42)
    ds = ds.map(lambda r, y: (cargar(r), y), num_parallel_calls=tf.data.AUTOTUNE)
    if entrenar:
        ds = ds.map(lambda x, y: (nitidez_aleatoria(aumentar(x, training=True)), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(LOTE).prefetch(tf.data.AUTOTUNE)

ds_ent, ds_val, ds_pru = conjunto(ent, True), conjunto(val, False), conjunto(pru, False)

pesos = compute_class_weight("balanced", classes=np.arange(len(CLASES)),
                             y=[CLASES.index(c) for c in ent[COLUMNA]])
pesos = dict(enumerate(pesos))

base = tf.keras.applications.MobileNetV2(input_shape=(TAM, TAM, 3), include_top=False,
                                         weights="imagenet", pooling="avg")
base.trainable = False
entrada = tf.keras.Input((TAM, TAM, 3))
x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(entrada)  # preprocesado de MobileNetV2 dentro del modelo
x = base(x, training=False)
x = tf.keras.layers.Dropout(0.3)(x)
salida = tf.keras.layers.Dense(len(CLASES), activation="softmax")(x)
modelo = tf.keras.Model(entrada, salida)

# Etapa 1: solo la capa nueva
modelo.compile(tf.keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", ["accuracy"])
parar = tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)
h1 = modelo.fit(ds_ent, validation_data=ds_val, epochs=15, class_weight=pesos, callbacks=[parar])

# Etapa 2: también las últimas capas de la red (transferencia de aprendizaje)
base.trainable = True
for capa in base.layers[:-30]:
    capa.trainable = False
modelo.compile(tf.keras.optimizers.Adam(1e-5), "sparse_categorical_crossentropy", ["accuracy"])
h2 = modelo.fit(ds_ent, validation_data=ds_val, epochs=15, class_weight=pesos, callbacks=[parar])

## 4. Evaluar en el conjunto de prueba

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, recall_score

y_real = np.array([CLASES.index(c) for c in pru[COLUMNA]])
y_pred = np.argmax(modelo.predict(ds_pru), axis=1)

print(classification_report(y_real, y_pred, labels=range(len(CLASES)), target_names=CLASES, digits=3, zero_division=0))
cm = confusion_matrix(y_real, y_pred, labels=range(len(CLASES)))
ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(xticks_rotation=30, cmap="Blues")
plt.title("Matriz de confusión · conjunto de prueba")
plt.tight_layout(); plt.savefig("matriz_confusion.png", dpi=150); plt.show()

## 5. Comparación con la regla sin IA (Tabla 7: «comparada con una versión sin IA»)

La regla de diferencia de píxeles solo sabe decir `pagina_nueva` o `misma_pagina`: da por buenas las
dos hojas y las hojas dobladas. Se evalúa en **el mismo** conjunto de prueba.

In [ ]:
y_regla = np.array([CLASES.index(c) for c in pru["regla_pixeles"]])
print("REGLA DE PÍXELES")
print(classification_report(y_real, y_regla, labels=range(len(CLASES)), target_names=CLASES, digits=3, zero_division=0))

i_dos = CLASES.index("dos_hojas")
tabla = pd.DataFrame({
    "F1 promedio": [f1_score(y_real, y_pred, average="macro"), f1_score(y_real, y_regla, average="macro")],
    "Recall dos_hojas": [recall_score(y_real, y_pred, labels=[i_dos], average="macro", zero_division=0),
                         recall_score(y_real, y_regla, labels=[i_dos], average="macro", zero_division=0)],
    "Exactitud": [np.mean(y_pred == y_real), np.mean(y_regla == y_real)],
}, index=["Verificador IA", "Regla de píxeles"]).round(3)
display(tabla)

f1 = tabla.loc["Verificador IA", "F1 promedio"]
rec = tabla.loc["Verificador IA", "Recall dos_hojas"]
print(f"Meta F1 ≥ 0,90: {'CUMPLE' if f1 >= 0.90 else 'NO CUMPLE'} ({f1:.3f})")
print(f"Meta recall dos_hojas ≥ 0,95: {'CUMPLE' if rec >= 0.95 else 'NO CUMPLE'} ({rec:.3f})")

## 6. Exportar a TensorFlow Lite y llevarlo a la Pi

Copie `verificador.tflite` y `verificador.txt` a `pi/modelos/` en la Raspberry Pi
(por ejemplo `scp verificador.* pi@archilector.local:~/ArchiLector/pi/modelos/`) y reinicie el servicio.
El panel web mostrará «Verificador: modelo entrenado». Mida el tiempo por volteo en la Pi (prueba T5.3, meta ≤ 1 s).

In [ ]:
# TensorFlow Lite cuantizado a 8 bits (pesos y activaciones), con entrada y
# salida en float32: la Pi le pasa los píxeles 0..255 tal cual.
def representativo():
    for x, _ in ds_ent.unbatch().take(200):
        yield [tf.cast(x[None], tf.float32)]

conv = tf.lite.TFLiteConverter.from_keras_model(modelo)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = representativo
tflite = conv.convert()
open(f"{NOMBRE}.tflite", "wb").write(tflite)
open(f"{NOMBRE}.txt", "w").write("\n".join(CLASES))  # el orden de las clases viaja con el modelo
print(f"{NOMBRE}.tflite: {len(tflite) / 1e6:.1f} MB")

# Comprobación: el modelo cuantizado acierta lo mismo que el original
interp = tf.lite.Interpreter(model_content=tflite)
interp.allocate_tensors()
ent_d, sal_d = interp.get_input_details()[0], interp.get_output_details()[0]
aciertos = 0
for x, y in ds_pru.unbatch():
    interp.set_tensor(ent_d["index"], tf.cast(x[None], ent_d["dtype"]))
    interp.invoke()
    aciertos += int(np.argmax(interp.get_tensor(sal_d["index"])[0]) == int(y))
print(f"Exactitud TFLite en prueba: {aciertos / len(pru):.3f}")

from google.colab import files
for f in (f"{NOMBRE}.tflite", f"{NOMBRE}.txt", "matriz_confusion.png"):
    files.download(f)